In [ ]:
from scipy.spatial import distance
import csv
import gensim
import hdbscan
import logging
import numpy as np
import random
from typing import Dict, Set, Tuple

logging.basicConfig(level=logging.WARN)

CENTROIDS_FILE = '/home/data/manisha/Manisha_Code/Updated_Code/clustering/centroids.csv'
SAMPLE_SIZE = 10000
DATA_VOIDS_2012_2017 = ['retard', 'russiagate', 'trumprussia','obamascare','lizardmen','chemtrails','coinflipgate','euthanasia','coinflipgate', 'hixenbaugh','mcmahons','pepe', 'slovenry', 'emailgate', 'benghazii', 'magathread', 'gerrymandering']

In [ ]:
MODEL_2012_2017 = gensim.models.Word2Vec.load('/home/data/manisha/Manisha_Code/Updated_Code/word2vec_models/data_word2vec_2012_2017.model')

In [ ]:
class CentroidExtractor:
    """
    A class to extract centroids from a gensim.models.Word2Vec model.
    :param cluster_labels: words corresponding to each label in the cluster.
    :param word_vectors: vectors corresponding to each word.
    """

    cluster_labels: Dict[int, Set[str]] = {} 
    word_vectors: Dict[str, np.ndarray] = {}

    @classmethod
    def get_word_label(cls, word: str, cluster_labels: dict = None) -> str:
        """
        Returns the label for a given word.
        :param word: 
        :param cluster_labels: 
        :return: 
        """
        if not cluster_labels:
            cluster_labels = cls.cluster_labels
        for label in cluster_labels.keys():
            if word in cluster_labels[label]:
                return label
    
    @classmethod
    def extract_word_vectors_and_labeled_set(cls) -> None:
        """
        Extract word_vectors and labeled_set from `CENTROIDS_FILE`.
        :return: None
        """
        with open(CENTROIDS_FILE, 'r') as infile:
            reader = csv.reader(infile)
            for row in reader:
                word = row[0]
                label = row[1]
                try:
                    cls.cluster_labels[label].add(word)
                except KeyError:
                    cls.cluster_labels[label] = {word}
                cls.word_vectors[word] = MODEL_2012_2017.wv[word]

    @classmethod
    def cluster_fit(cls, words: list) -> Tuple[hdbscan.hdbscan_.HDBSCAN, list, np.ndarray]:
        """
        Fit a gensim.models.Word2Vec model to the given list of words.
        :param words: list of words returned from cluster_labels[label]
        :return: clusterer, words, vectors
        """
        number_of_words = len(words)
        logging.info(f"Number of words in this cluster: {number_of_words}")
        if number_of_words > SAMPLE_SIZE:
            random.shuffle(words)
            sample_words = words[:SAMPLE_SIZE]
        else:
            sample_words = words
        vectors = []
        for word in sample_words:
            vectors.append(cls.word_vectors[word])
        logging.info('Running HDBSCAN to cluster input words...')
        clusterer = hdbscan.HDBSCAN(min_cluster_size=2, gen_min_span_tree=True) 
        clusterer.fit(vectors)
        logging.info(f"Unique labels in this iteration: {set(clusterer.labels_)}")
        if set(clusterer.labels_) == {-1}: # if all the labels are -1
            logging.warning("All labels are -1")
        logging.info(f"Total words in this cluster: {len(clusterer.labels_)}")
        return clusterer, words, vectors
    
    @classmethod
    def cluster_relabel(cls, clusterer, words, vectors) -> Dict[int, Set[str]]:
        """
        Fitting all words to the existing clusters and relabeling them.
        :param clusterer: HDBScan cluster on words.
        :param words: cluster words.
        :param vectors: Vectors corresponding to each word.
        :return: 
        """
        if len(set(clusterer.labels_)) > 1:
            centroids_raw = {}
            labels = clusterer.labels_
            for i in range(0, len(labels)):
                if labels[i] != -1:
                    try:
                        centroids_raw[labels[i]].append(vectors[i])
                    except KeyError:
                        centroids_raw[labels[i]] = [vectors[i]]
            logging.info('Fitting all the words to the formed clusters...')
            centroid_mean = {}
            for key in centroids_raw:
                centroid_mean[key] = np.mean(centroids_raw[key], axis=0)
            new_cluster_labels = {}
            for word in words:
                distances = {}
                for center in centroid_mean:
                    distances[center] = distance.cosine(cls.word_vectors[word], centroid_mean[center])
                cluster_label = sorted(distances.items(), key=lambda x: x[1])[0][0]
                try:
                    new_cluster_labels[cluster_label].add(word)
                except KeyError:
                    new_cluster_labels[cluster_label] = {word}
            return new_cluster_labels
        raise ValueError("Cannot relabel labels <= 1.")

In [ ]:
class Centroid:
    """
    A class to represent a gensim.models.Word2Vec model.
    """
    def __init__(self, data_void: str, cluster_labels: dict = CentroidExtractor.cluster_labels):
        """
        :param data_void: 
        :param cluster_labels: 
        """
        self.data_void = data_void
        self.cluster_labels = cluster_labels
        self.label = CentroidExtractor.get_word_label(self.data_void, self.cluster_labels)
        self.next_centroid = None


In [ ]:
def find_closest_words(centroid: Centroid, level=0) -> None:
    """
    Recursively find the closest words to a given centroid.
    :param centroid: Centroid of the cluster.
    :param level: Level of each iteration
    :return: None
    """
    logging.info(f'Iteration level: {level}')
    logging.info(f'Running `find_closest_words` on centroid: {centroid.label}...')
    logging.info(f"Words in this iteration are: {centroid.cluster_labels[centroid.label]}")
    logging.info(f'Running the sub-clustering algorithm on words in this iteration...')
    clusterer, words, vectors = CentroidExtractor.cluster_fit(list(centroid.cluster_labels[centroid.label]))
    if set(clusterer.labels_) == {-1}:
        logging.info(f'Ending sub-clustering...')
        return
    logging.info(f'Relabeling the clusters...')
    new_cluster_labels = CentroidExtractor.cluster_relabel(clusterer, words, vectors)
    logging.info(f'Creating new centroid with the new cluster labels...')
    new_centroid = Centroid(
        data_void=centroid.data_void,
        cluster_labels=new_cluster_labels
    )
    centroid.next_centroid = new_centroid
    find_closest_words(new_centroid, level + 1)
    
data_void_clusters = {}
CentroidExtractor.extract_word_vectors_and_labeled_set()
for data_void in DATA_VOIDS_2012_2017[:1]:
    logging.info(f"------------------- DATA VOID TERM : {data_void} -------------------")
    root_centroid = Centroid(data_void)
    find_closest_words(root_centroid)

    head = root_centroid
    while head:
        if not head.next_centroid:
            data_void_clusters[head.data_void] = head.cluster_labels[head.label]
            print(head.data_void)
            print(head.cluster_labels[head.label])
        head = head.next_centroid

In [ ]:
data_void_clusters